In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master('local[*]').appName('test').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/26 00:29:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/26 00:29:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
df_green = spark.read.option('recursiveFileLookup', 'true').parquet('./data/pq/green')

In [3]:
df_green.columns

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [14]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']
duration_rdd = df_green\
    .repartition(4)\
    .select(columns)\
    .rdd

In [15]:
rows = duration_rdd.take(10)

In [16]:
rows

[Row(VendorID=1, lpep_pickup_datetime=datetime.datetime(2020, 2, 11, 19, 58, 36), PULocationID=145, DOLocationID=92, trip_distance=9.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 10, 15, 46, 4), PULocationID=174, DOLocationID=265, trip_distance=4.83),
 Row(VendorID=1, lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 8, 17, 15), PULocationID=235, DOLocationID=174, trip_distance=0.0),
 Row(VendorID=None, lpep_pickup_datetime=datetime.datetime(2020, 1, 1, 8, 14), PULocationID=169, DOLocationID=235, trip_distance=1.33),
 Row(VendorID=1, lpep_pickup_datetime=datetime.datetime(2020, 2, 11, 16, 43, 41), PULocationID=228, DOLocationID=149, trip_distance=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 15, 17, 30, 8), PULocationID=42, DOLocationID=116, trip_distance=0.73),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 14, 16, 52, 59), PULocationID=116, DOLocationID=132, trip_distance=19.85),
 Row(VendorID=1, lpep_pickup_datetime

In [25]:
import pandas as pd

df_test = pd.DataFrame(rows, columns=columns)

list(df_test.itertuples())

[Pandas(Index=0, VendorID=1.0, lpep_pickup_datetime=Timestamp('2020-02-11 19:58:36'), PULocationID=145, DOLocationID=92, trip_distance=9.0),
 Pandas(Index=1, VendorID=2.0, lpep_pickup_datetime=Timestamp('2020-01-10 15:46:04'), PULocationID=174, DOLocationID=265, trip_distance=4.83),
 Pandas(Index=2, VendorID=1.0, lpep_pickup_datetime=Timestamp('2020-01-14 08:17:15'), PULocationID=235, DOLocationID=174, trip_distance=0.0),
 Pandas(Index=3, VendorID=nan, lpep_pickup_datetime=Timestamp('2020-01-01 08:14:00'), PULocationID=169, DOLocationID=235, trip_distance=1.33),
 Pandas(Index=4, VendorID=1.0, lpep_pickup_datetime=Timestamp('2020-02-11 16:43:41'), PULocationID=228, DOLocationID=149, trip_distance=0.0),
 Pandas(Index=5, VendorID=2.0, lpep_pickup_datetime=Timestamp('2020-01-15 17:30:08'), PULocationID=42, DOLocationID=116, trip_distance=0.73),
 Pandas(Index=6, VendorID=2.0, lpep_pickup_datetime=Timestamp('2020-01-14 16:52:59'), PULocationID=116, DOLocationID=132, trip_distance=19.85),
 Pa

In [23]:
def model_predict(df):
    y_pred = df.trip_distance * 5
    return y_pred

In [26]:
def apply_model_in_batch(rows):

    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row

In [28]:
df_predicts = duration_rdd\
    .mapPartitions(apply_model_in_batch)\
    .toDF()\
    .drop('Index')

In [31]:
df_predicts.rdd.getNumPartitions()

4

In [32]:
df_predicts.show()

+--------+--------------------+------------+------------+-------------+------------------+
|VendorID|lpep_pickup_datetime|PULocationID|DOLocationID|trip_distance|predicted_duration|
+--------+--------------------+------------+------------+-------------+------------------+
|     1.0|                  {}|         145|          92|          9.0|              45.0|
|     2.0|                  {}|         174|         265|         4.83|             24.15|
|     1.0|                  {}|         235|         174|          0.0|               0.0|
|     NaN|                  {}|         169|         235|         1.33|              6.65|
|     1.0|                  {}|         228|         149|          0.0|               0.0|
|     2.0|                  {}|          42|         116|         0.73|              3.65|
|     2.0|                  {}|         116|         132|        19.85|             99.25|
|     1.0|                  {}|          97|          35|          0.0|               0.0|

Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/Applications/miniconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [1]:
spark.stop()

NameError: name 'spark' is not defined